# Análise de Dados da Google Play Store

Este notebook irá realizar o refinamento e análise dos dados de aplicativos da **Google Play Store**. O primeiro passo é deixar os dados aptos a serem trabalhados, para isso é necessário realizar a leitura dos dados e a Remoção de linhas duplicadas.

## Leitura dos Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tabela = pd.read_csv("googleplaystore.csv", sep=",")
print(f'Número de linhas: {len(tabela)}')
tabela.head()

## Remoção de Linhas Duplicadas

In [ ]:
print("Antes:", tabela.shape)

tabela = tabela.drop_duplicates()

print("Depois:", tabela.shape)
display(tabela)

---

## Top 5 Apps por Número de Instalações

Primeiro passo é garantir que a tabela 'Installs' esta em formato numérico, a fim de melhor tratar os dados

In [ ]:
tabela['Installs'] = tabela['Installs'].astype(str)
tabela['Installs'] = tabela['Installs'].str.replace('[+,]', '', regex=True)
tabela['Installs'] = pd.to_numeric(tabela['Installs'], errors='coerce')

Remoção de duplicidade, a fim de não aparecer o mesmo apps em versões diferentes

In [ ]:
tabela_unica = tabela.sort_values(by='Installs', ascending=False).drop_duplicates(subset='App', keep='first')

Seleção dos top 5 apps

In [ ]:
top5_unicos = tabela_unica[['App', 'Installs']].head(5)

Gráfico

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(top5_unicos['App'], top5_unicos['Installs'], color='mediumseagreen')
plt.title('Top 5 Apps com Mais Instalações (Nomes Únicos)')
plt.xlabel('App')
plt.ylabel('Instalações')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Apps por Categoria

Contagem do numero de apps por categoria

In [ ]:
categorias = tabela['Category'].value_counts()

Gráfico

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(categorias[:10],
        labels=categorias.index[:10],
        autopct='%1.1f%%',
        startangle=140,
        colors=plt.cm.Set3.colors)

plt.title('Distribuição dos Apps por Categoria (Top 10)')
plt.axis('equal')
plt.tight_layout()
plt.show()

## App Mais Caro

Garantindo que a tabela 'Price' esta em formato numérico

In [ ]:
tabela['Price'] = tabela['Price'].astype(str).str.replace('$', '', regex=False)
tabela['Price'] = pd.to_numeric(tabela['Price'], errors='coerce')

Garantindo que a tabela 'Installs' esta em formato numérico

In [ ]:
tabela['Installs'] = tabela['Installs'].astype(str).str.replace('[+,]', '', regex=True)
tabela['Installs'] = pd.to_numeric(tabela['Installs'], errors='coerce')

Leitura do app mais caro

In [ ]:
app_mais_caro = tabela.sort_values(by='Price', ascending=False).iloc[0]

Leitura da renda estimada feita por ele

In [ ]:
renda_estimada = app_mais_caro['Price'] * app_mais_caro['Installs']

Resultado

In [ ]:
print("App mais caro:")
print(app_mais_caro[['App', 'Price', 'Installs']])
print(f"\nReceita estimada: ${renda_estimada:,.2f}")

## Apps classificados como 'Mature 17+'?

Criando uma tabela so com os app cassificados como 'Mature 17+'

In [ ]:
mature_apps = tabela[tabela['Content Rating'] == 'Mature 17+']
display(mature_apps[['App', 'Category', 'Rating', 'Installs']].head(10))

Mostrando o numero de apps com essa classificação, os tops 5 app instalados e o numero de apps por categoria

In [ ]:
print(f"\nTotal de apps classificados como 'Mature 17+': {len(mature_apps)}")

mature_ordenado = mature_apps.sort_values(by='Installs', ascending=False)
top5_mature_unicos = mature_ordenado.drop_duplicates(subset='App', keep='first').head(5)
display(top5_mature_unicos[['App', 'Installs']])

print(mature_apps['Category'].value_counts())


## Top 10 Apps por Reviews

Garantindo que a tabela 'Reviews' esta em formato numérico

In [ ]:
tabela['Reviews'] = pd.to_numeric(tabela['Reviews'], errors='coerce')

Organizando por numero de reviews e analizando o top 10

In [ ]:
ordenado_por_reviews = tabela.sort_values(by='Reviews', ascending=False)
top10_reviews_unicos = ordenado_por_reviews.drop_duplicates(subset='App', keep='first').head(10)

Resultado

In [ ]:
display(top10_reviews_unicos[['App', 'Reviews']])

---

## App com Mais Reviews das Top 10 categorias

Garantindo que a tabela 'Reviews' esta em formato numérico

In [ ]:
tabela['Reviews'] = pd.to_numeric(tabela['Reviews'], errors='coerce')

Organizando os Apps por numero de reviews e analizando as top 10 Categorias

In [ ]:
mais_reviews_por_categoria = tabela.sort_values(by='Reviews', ascending=False).dropna(subset=['Category'])
top_por_categoria = mais_reviews_por_categoria.drop_duplicates(subset='Category', keep='first')

In [ ]:
top10 = top_por_categoria.sort_values(by='Reviews', ascending=False).head(10)

In [ ]:
categorias = top10['Category']
reviews = top10['Reviews']
nomes_apps = top10['App']

Resultado

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(categorias, reviews, color='mediumslateblue')
plt.xlabel('Número de Reviews')
plt.title('App com Mais Reviews das Top 10 categorias')
plt.gca().invert_yaxis()

for i, (review, nome) in enumerate(zip(reviews, nomes_apps)):
    plt.text(review, i, f"{nome}", va='center', fontsize=8)

plt.tight_layout()
plt.show()

## As Top 7 categorias com maior lucro médio

Limpeza

In [ ]:
tabela['Price'] = tabela['Price'].astype(str).str.replace('$', '', regex=False)
tabela['Price'] = pd.to_numeric(tabela['Price'], errors='coerce')

tabela['Installs'] = tabela['Installs'].astype(str).str.replace('[+,]', '', regex=True)
tabela['Installs'] = pd.to_numeric(tabela['Installs'], errors='coerce')

Calculo da receita por app, da média de lucro por categoria e das Top 7 categorias com maior lucro médio

In [ ]:
tabela['Receita'] = tabela['Price'] * tabela['Installs']

media_receita_categoria = tabela.groupby('Category')['Receita'].mean().sort_values(ascending=False)

top7_lucro_medio = media_receita_categoria.head(7)

Resultado

In [ ]:

plt.figure(figsize=(10,6))
plt.barh(top7_lucro_medio.index, top7_lucro_medio.values, color='seagreen')
plt.xlabel('Lucro Médio Estimado (US$)')
plt.title('Top 7 Categorias com Maior Lucro Médio por App')
plt.gca().invert_yaxis()

for i, valor in enumerate(top7_lucro_medio.values):
    plt.text(valor, i, f"${valor:,.0f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()


## As Top 7 categorias que mais lucraram com maior Numero de reviews

Limpeza

In [ ]:
tabela['Price'] = tabela['Price'].astype(str).str.replace('$', '', regex=False)
tabela['Price'] = pd.to_numeric(tabela['Price'], errors='coerce')

tabela['Installs'] = tabela['Installs'].astype(str).str.replace('[+,]', '', regex=True)
tabela['Installs'] = pd.to_numeric(tabela['Installs'], errors='coerce')

tabela['Reviews'] = pd.to_numeric(tabela['Reviews'], errors='coerce')

Cálculos

In [ ]:
tabela['Receita'] = tabela['Price'] * tabela['Installs']

top7_reviews = (
    tabela.groupby('Category')['Reviews']
    .sum()
    .sort_values(ascending=False)
    .head(7)
    .index
)

tabela_top7_reviews = tabela[tabela['Category'].isin(top7_reviews)]

lucro_medio_top7 = (
    tabela_top7_reviews.groupby('Category')['Receita']
    .mean()
    .sort_values(ascending=False)
)

Resultado

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(lucro_medio_top7.index, lucro_medio_top7.values, color='orange')
plt.xlabel('Lucro Médio Estimado (US$)')
plt.title('Média de Lucro nas 7 Categorias com Mais Reviews')
plt.gca().invert_yaxis()

for i, valor in enumerate(lucro_medio_top7.values):
    plt.text(valor, i, f"${valor:,.0f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Apps Gratuitos vs Pagos

In [ ]:

plt.figure(figsize=(6, 6))
wedges, texts, autotexts = plt.pie(
    media_reviews,
    labels=media_reviews.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=cores,
    pctdistance=0.85
)
centro = plt.Circle((0, 0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centro)

plt.title('Média de Reviews: Apps Gratuitos vs Pagos \n')
plt.axis('equal')
plt.tight_layout()
plt.show()